In [5]:
import os
from langchain_community.document_loaders import TextLoader

C:\Users\komal\AppData\Local\Temp\ipykernel_18488\2530354140.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [6]:
import re
def clean_markdown(text):

    # Remove YAML front matter
    text = re.sub(
        r"\A---\s*\n.*?\n---\s*\n?",
        "",
        text,
        flags=re.DOTALL
    )

    # Remove GitLab/Jekyll shortcodes
    text = re.sub(
        r"\{\{%.*?%\}\}",
        "",
        text,
        flags=re.DOTALL
    )

    # Remove empty template placeholders
    text = re.sub(
        r"\{\{\s*\}\}",
        "",
        text
    )

    # Remove HTML tags
    text = re.sub(
        r"<[^>]+>",
        "",
        text
    )

    # Convert Markdown links to visible text
    text = re.sub(
        r"\[([^\]]+)\]\([^)]+\)",
        r"\1",
        text
    )

    # Remove horizontal rules
    text = re.sub(
        r"(?m)^\s*([-*_]){3,}\s*$",
        "",
        text
    )

    # Remove obvious stray underscore at the end
    text = re.sub(
        r"_\s*$",
        "",
        text
    )

    # Remove excessive blank lines
    text = re.sub(
        r"\n{3,}",
        "\n\n",
        text
    )

    return text.strip()




In [7]:
def load_all_md():
    folder_path = "data/pdfs"   # <-- points to where company/finance/people_group actually live
    num_docs = 0
    all_docs = []
    
    for root, dirs, files in os.walk(folder_path):
        # Skip Jupyter checkpoint folders entirely
        dirs[:] = [d for d in dirs if d != ".ipynb_checkpoints"]
        
        for filename in files:
            if filename.lower().endswith(".md"):
                md_path = os.path.join(root, filename)
                
                loader = TextLoader(md_path, encoding="utf-8")
                doc = loader.load()[0]
                
                rel_path = os.path.relpath(md_path, folder_path)
                department = rel_path.split(os.sep)[0]
                
                doc.metadata["source"] = rel_path
                doc.metadata["department"] = department
                doc.metadata["filename"] = filename
                doc.metadata["version_ts"] = os.path.getmtime(md_path)
                
                original_text = doc.page_content

                cleaned_text = clean_markdown(original_text)
                
                if "{{}}" in original_text:
                    print("FOUND IN RAW:", md_path)
                    print("RAW:", repr(original_text[:200]))
                    print("CLEANED:", repr(cleaned_text[:200]))
                    print("CLEAN AGAIN:", repr(clean_markdown("{{}}")))
                
                doc.page_content = cleaned_text
                all_docs.append(doc)
                num_docs += 1
    
    print("total mds:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [8]:
documents=load_all_md()

total mds: 27
total pages: 27


In [9]:
for doc in documents:
    if "{{}}" in doc.page_content:
        print("❌ FOUND:", doc.metadata["source"])

❌ FOUND: company\team-pets.md
❌ FOUND: people_group\acceptable-use-policy.md
❌ FOUND: people_group\contracts-probation-periods.md
❌ FOUND: people_group\guidance-on-feedback.md
❌ FOUND: people_group\visas.md
❌ FOUND: people_group\offboarding\offboarding_standards.md


In [10]:
# def update_heading_hierarchy(heading_stack, line):
#     """
#     Update Markdown heading hierarchy.

#     Example:

#     # Engineering
#     ## Testing
#     ### Unit Testing

#     Result:
#     Engineering > Testing > Unit Testing
#     """

#     # Count heading level
#     level = len(line) - len(line.lstrip("#"))

#     # Remove # symbols and spaces
#     heading = line.lstrip("#").strip()

#     # Remove current level and deeper levels
#     heading_stack = heading_stack[:level - 1]

#     # Add current heading
#     heading_stack.append(heading)

#     return heading_stack

In [11]:
# import re

# def split_long_paragraph(para, chunk_size, chunk_overlap):
#     """
#     Split a large paragraph into smaller chunks
#     while trying to preserve complete sentences.
#     """

#     # Split paragraph into sentences
#     sentences = re.split(r'(?<=[.!?])\s+', para)

#     chunks = []
#     current = ""

#     for sentence in sentences:

#         sentence = sentence.strip()

#         if not sentence:
#             continue

#         # Try adding the sentence to the current chunk
#         if len(current) + len(sentence) + 1 <= chunk_size:

#             current = (
#                 current + " " + sentence
#             ).strip()

#         else:

#             # Current chunk is complete
#             if current:
#                 chunks.append(current)

#             # If the sentence itself is too large
#             if len(sentence) > chunk_size:

#                 start = 0

#                 while start < len(sentence):

#                     end = start + chunk_size

#                     chunks.append(
#                         sentence[start:end]
#                     )

#                     start += chunk_size - chunk_overlap

#                 current = ""

#             else:

#                 # Start a new chunk with this sentence
#                 current = sentence

#     # Add remaining text
#     if current:
#         chunks.append(current)

#     return chunks

In [12]:

# def parse_sections(text):
#     """
#     Convert Markdown text into logical sections.

#     Each section contains:
#     - heading
#     - section hierarchy
#     - content
#     """

#     sections = []

#     heading_stack = []

#     current_heading = ""
#     current_section = ""

#     current_content = []

#     # Process document line by line
#     lines = text.splitlines()

#     for line in lines:

#         line = line.strip()

#         # Ignore empty lines
#         if not line:
#             continue

#         # --------------------------------
#         # Check whether line is a heading
#         # --------------------------------

#         if re.match(r"^#{1,6}\s+", line):

#             # Save previous section
#             if current_content:

#                 sections.append({
#                     "heading": current_heading,
#                     "section": current_section,
#                     "content": "\n".join(
#                         current_content
#                     ).strip()
#                 })

#             # Update heading hierarchy
#             heading_stack = update_heading_hierarchy(
#                 heading_stack,
#                 line
#             )

#             # Create hierarchy string
#             current_section = " > ".join(
#                 heading_stack
#             )

#             # Store current heading
#             current_heading = line

#             # Start collecting new section content
#             current_content = []

#         else:

#             # Normal content
#             current_content.append(line)

#     # --------------------------------
#     # Save the final section
#     # --------------------------------

#     if current_content:

#         sections.append({
#             "heading": current_heading,
#             "section": current_section,
#             "content": "\n".join(
#                 current_content
#             ).strip()
#         })

#     return sections

In [13]:
# def chunk_documents(
#     documents,
#     chunk_size=800,
#     chunk_overlap=150
# ):
#     """
#     Create final RAG chunks from documents.
#     """

#     all_chunks = []
#     chunk_id = 0

#     for doc in documents:

#         # -----------------------------------------
#         # Convert document into logical sections
#         # -----------------------------------------

#         sections = parse_sections(
#             doc.page_content
#         )

#         for section in sections:

#             heading = section["heading"]
#             section_name = section["section"]
#             content = section["content"]

#             if not content:
#                 continue

#             # -----------------------------------------
#             # Keep heading with its content
#             # -----------------------------------------

#             if heading:

#                 section_text = (
#                     heading
#                     + "\n\n"
#                     + content
#                 )

#             else:

#                 section_text = content

#             # -----------------------------------------
#             # Case 1:
#             # Section fits within chunk size
#             # -----------------------------------------

#             if len(section_text) <= chunk_size:

#                 all_chunks.append({

#                     "chunk_id": chunk_id,

#                     "text": section_text,

#                     "source": doc.metadata["source"],

#                     "department": doc.metadata["department"],

#                     "filename": doc.metadata["filename"],

#                     "version_ts": doc.metadata["version_ts"],

#                     "section": section_name
#                 })

#                 chunk_id += 1

#             # -----------------------------------------
#             # Case 2:
#             # Section is too large
#             # -----------------------------------------

#             else:

#                 # Reserve space for heading
#                 available_size = (
#                     chunk_size - len(heading) - 2
#                     if heading
#                     else chunk_size
#                 )

#                 sentence_chunks = split_long_paragraph(
#                     content,
#                     available_size,
#                     chunk_overlap
#                 )

#                 for i, chunk_text in enumerate(
#                     sentence_chunks
#                 ):

#                     # Add heading to first chunk
#                     if i == 0 and heading:

#                         final_text = (
#                             heading
#                             + "\n\n"
#                             + chunk_text
#                         )

#                     else:

#                         final_text = chunk_text

#                     all_chunks.append({

#                         "chunk_id": chunk_id,

#                         "text": final_text,

#                         "source": doc.metadata["source"],

#                         "department": doc.metadata["department"],

#                         "filename": doc.metadata["filename"],

#                         "version_ts": doc.metadata["version_ts"],

#                         "section": section_name
#                     })

#                     chunk_id += 1

#     # -----------------------------------------
#     # Statistics
#     # -----------------------------------------

#     print("Total chunks:", len(all_chunks))

#     if all_chunks:

#         lengths = [
#             len(chunk["text"])
#             for chunk in all_chunks
#         ]

#         print("Min length:", min(lengths))
#         print("Max length:", max(lengths))

#         print(
#             "Average length:",
#             sum(lengths) / len(lengths)
#         )

#         print(
#             "Chunks below 200:",
#             sum(
#                 1
#                 for length in lengths
#                 if length < 200
#             )
#         )

#     return all_chunks

In [14]:
# for chunk in chunks[:5]:

#     print("\n--- CHUNK ---")

#     print("ID:", chunk["chunk_id"])

#     print("Length:", len(chunk["text"]))

#     print("Section:", chunk["section"])

#     print("Source:", chunk["source"])

#     print("Text:")
#     print(chunk["text"])

In [15]:
import re


def is_heading(line):
    # FIX 1: a real markdown heading is 1-6 '#' followed by a space.
    # This ignores things like "#hashtag" or "#!/bin/bash".
    return re.match(r"^#{1,6}\s+\S", line) is not None


def update_heading_hierarchy(heading_stack, line):
    """
    Update Markdown heading hierarchy.
    Example: Engineering > Testing > Integration Testing
    """
    level = len(line) - len(line.lstrip("#"))
    heading = line.lstrip("#").strip()

    # Remove headings at the same or deeper level
    heading_stack = heading_stack[:level - 1]

    # Add the new heading
    heading_stack.append(heading)

    return heading_stack


def split_long_paragraph(para, chunk_size, chunk_overlap):
    """
    Split a paragraph while trying to preserve complete sentences.
    (unchanged from your version)
    """
    sentences = re.split(r'(?<=[.!?])\s+', para)

    chunks = []
    current = ""

    for sentence in sentences:
        sentence = sentence.strip()

        if not sentence:
            continue

        if len(current) + len(sentence) + 1 <= chunk_size:
            current = (current + " " + sentence).strip()
        else:
            if current:
                chunks.append(current)

            if len(sentence) > chunk_size:
                start = 0
                while start < len(sentence):
                    chunks.append(sentence[start:start + chunk_size])
                    start += chunk_size - chunk_overlap
                current = ""
            else:
                current = sentence

    if current:
        chunks.append(current)

    return chunks


def make_chunk(doc, chunk_id, text, section):
    # FIX 4: one place that builds a chunk, so you don't repeat the dict 3 times.
    return {
        "chunk_id": chunk_id,
        "text": text,                                     # show this to the user
        "embed_text": (section + "\n\n" + text) if section else text,  # embed THIS
        "source": doc.metadata["source"],
        "title": doc.metadata.get("title", doc.metadata["filename"]),
        "department": doc.metadata["department"],
        "filename": doc.metadata["filename"],
        "version_ts": doc.metadata.get("version_ts"),
        "section": section,
    }


def chunk_documents(documents, chunk_size=800, chunk_overlap=150):
    all_chunks = []

    for doc in documents:
        text = doc.page_content
        source = doc.metadata["source"]

        n = 0                      # FIX 3: chunk number inside THIS document
        heading_stack = []
        in_code = False            # FIX 1: are we inside a ``` code block?

        paragraphs = text.split("\n\n")
        current = ""
        current_section = ""       # section of the paragraph we are reading now
        chunk_section = ""         # FIX 2: section where the current chunk STARTED

        for para in paragraphs:
            para = para.strip()

            if not para:
                continue

            for line in para.split("\n"):
                line = line.strip()

                if line.startswith("```"):
                    in_code = not in_code          # entering/leaving a code block
                elif not in_code and is_heading(line):
                    heading_stack = update_heading_hierarchy(heading_stack, line)
                    current_section = " > ".join(heading_stack)

            # Add paragraph to current chunk
            if len(current) + len(para) + 2 <= chunk_size:
                if not current:
                    chunk_section = current_section   # this para starts a new chunk
                current = (current + "\n\n" + para).strip()

            else:
                if current:
                    all_chunks.append(
                        make_chunk(doc, f"{source}::{n}", current, chunk_section)
                    )
                    n += 1
                    current = ""

                if len(para) <= chunk_size:
                    current = para
                    chunk_section = current_section   # new chunk starts here
                else:
                    for chunk_text in split_long_paragraph(
                        para, chunk_size, chunk_overlap
                    ):
                        all_chunks.append(
                            make_chunk(doc, f"{source}::{n}", chunk_text, current_section)
                        )
                        n += 1

        # Add remaining chunk
        if current:
            all_chunks.append(
                make_chunk(doc, f"{source}::{n}", current, chunk_section)
            )
            n += 1

    print("total chunks:", len(all_chunks))
    return all_chunks


In [16]:
chunks = chunk_documents(documents)

total chunks: 1024


In [17]:
for chunk in chunks[:5]:

    print("\n--- CHUNK ---")

    print("ID:", chunk["chunk_id"])

    print("Length:", len(chunk["text"]))

    print("Section:", chunk["section"])

    print("Source:", chunk["source"])

    print("Text:")
    print(chunk["text"])


--- CHUNK ---
ID: company\cadence.md::0
Length: 784
Section: Overview
Source: company\cadence.md
Text:
## Overview

Everything in a company happens in a certain cadence.
The period of each cadence differs.
The timescale between periods are about 4x, varying from 3x to 5x.
Below are the cadences we have at GitLab:

1. 30 years (3x 10 years)
1. 10 years (3.3x 3-years)
1. 3 years (3x year)
1. Year (4x quarter)
1. Quarter (3x month)
1. Month (4.3x week)
1. Week

Items on this page are grouped into a cadence based on the underlying period of time that the item pertains to, not based on when the item is updated. For example, our strategy looks three years out but is reviewed annually by E-Group and may be updated more frequently if the need arises.

A overview of key company dates for FY25 can be found here

### Cadence Examples

How elements of our cadence fit together over time:

--- CHUNK ---
ID: company\cadence.md::1
Length: 593
Section: Overview > Cadence Examples
Source: company\caden

In [18]:
# !pip install sentence-transformers

Defaulting to user installation because normal site-packages is not writeable
